# Impact Labelling

This notebook serves as guidelines to label qualitative and quantitative impact information \
Other guidelines can be found here : https://docs.google.com/document/d/1SZk7cou4R6yV44UAxehUqOdqndOFiyHbUCUptlwkk_w/edit?tab=t.0#heading=h.2s6r53ihrgdp

Recommandations
- Work at the sentence level. Each time you identify one or several impacts in a sentence, extract all the information from it. Then continue looping over text sentence
- Information on the date and location can be present in different sentences like in sentences discribing the hazard, but sure not to miss them. 

In [6]:
import pandas as pd
import json
from collections import Counter
import pandas as pd
from src.LLM_functions import *
import copy as cp
import matplotlib.pyplot as plt
import json_repair

In [7]:
#helper functions
def take_latest_report(df, date_field="reportDate"):
    """
    Select the most recent report for each appeal code.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing the reports to be filtered.
    date_field : str, optional
        Field in the DataFrame containing the dates of the reports.
        Default is "reportDate".

    Returns
    -------
    filtered_df : pandas.DataFrame
        DataFrame containing the most recent reports for each appeal code.
    """
    df_out = (
            df.groupby("appealCode", as_index=False)
            .apply(lambda x: x.sort_values(date_field, ascending=False).head(1))
            .reset_index(drop=True)
        )
    return df_out

def take_longest_report(df):
    """
    Select the report with the longest nathaz_text for each appeal code.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing the reports to be filtered.

    Returns
    -------
    filtered_df : pandas.DataFrame
        DataFrame containing the filtered reports.
    """
    df_out = (
            df.groupby("appealCode", as_index=False)
            .apply(lambda x: x.loc[x.nathaz_text.str.len().idxmax()])
            .reset_index(drop=True)
        )
    return df_out

def print_report(appeal_code, report, text_field="nathaz_text"):
    print(f"{appeal_code}: {report.date}")
    text = report[text_field]
    print("\n".join(text))

def add_report_date(report, impact_dict_or_list):
    if isinstance(impact_dict_or_list, dict):
        impact_dict_or_list["reportDate"] = report.date
    elif isinstance(impact_dict_or_list, list):
        for i in range(len(impact_dict_or_list)):
            impact_dict_or_list[i]["reportDate"] = report.date
    else:
        raise TypeError("impact_dict_or_list must be a dictionary or a list of dictionaries")
    return impact_dict_or_list

def download_report(report, savelocation):
    link = report["reportLink"]
    savename = report["origType"]+".pdf"
    r = requests.get(link)
    with open(savelocation + savename, 'wb') as f:
        f.write(r.content)

def report_dict_to_df(labelled_reports_dict):
    df_list = []
    for k,v in labelled_reports_dict.items():
        df = pd.DataFrame(v)
        df['appealCode'] = k
        df_list.append(df)
    df_all = pd.concat(df_list)
    df_all.reset_index(inplace=True, drop=True)
    return df_all

def listify_strings(x):
    """
    Convert strings to lists if possible

    If a string can be parsed as a list, it is converted to a list. If not,
    the string is wrapped in a list. If the input is already a list, it is
    returned as is. If the input is None or NaN, an empty list is returned.

    Parameters
    ----------
    x : str or list or None or numpy.nan
        Element to be converted

    Returns
    -------
    list
        The converted element
    """
    if isinstance(x, str):
        try :
            x = json_repair.loads(x)
        except :
            x = [x]
        return x
    elif isinstance(x, list):
        return x
    elif pd.isna(x) or x is None:
        return []
    else:
        return x

In [8]:
### Paths
# DATA_IN_JSONS = "..." #### CHANGE
# DATA_LABELLED = "..." #### CHANGE 

In [ ]:
#Load data
file_path = DATA_IN_JSONS / 'preproc_text_sel_gaps_df_with_clean_text_all_v190526.csv'

# Open from csv
filtered_reports = pd.read_csv(file_path)

filtered_reports = take_latest_report(filtered_reports)

C:\Users\lhasbini\AppData\Local\Temp\ipykernel_25116\1520472424.py:42: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.loc[x.nathaz_text.str.len().idxmax()])


## Prepare data

In [ ]:
# select reports to be labelled
# appealCode_list = ["MDRBD022"]  #### UPDATE HERE WITH THE LIST OF APPEAL CODE THAT NEED TO BE LABELLED
appealCode_list = [
    "MDRIR009", "MDRTP004", 
    "MDRVU012", "MDRDZ007",
    "MDRAF014", "MDRCM036",
    "MDRDZ008", "MDRPH036",
    "MDRHT018", "MDRID013",
    "MDR55001", "MDRJO003",
    "MDRLB013", "MDRPK018",
    "MDRTJ035", "MDRRS015",
    "MDRMN017", "MDRPG008",
    "MDREC019", "MDRKZ010", "MDRUY004"]  #### UPDATE HERE WITH THE LIST OF APPEAL CODE THAT NEED TO BE LABELLED

reports_to_label_all = filtered_reports.loc[filtered_reports["appealCode"].isin(appealCode_list)]#filtered_reports.where(filtered_reports.appealCode.isin(appealCode_list))#.dropna()

#convert dates
reports_to_label_all.date = pd.to_datetime(reports_to_label_all.reportDate, dayfirst=True)

# ensure `nathaz_text` is a list of sentence strings
col_to_list = ["nathaz_text"]
reports_to_label_all[col_to_list] = reports_to_label_all[col_to_list].map(lambda x: listify_strings(x))

#check that everything is there
print(f"Number appealCodes: {len(appealCode_list)}, number reports: {len(reports_to_label_all)}")

C:\Users\lhasbini\AppData\Local\Temp\ipykernel_25116\71458790.py:18: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  reports_to_label_all.date = pd.to_datetime(reports_to_label_all.reportDate, dayfirst=True)
C:\Users\lhasbini\AppData\Local\Temp\ipykernel_25116\71458790.py:18: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  reports_to_label_all.date = pd.to_datetime(reports_to_label_all.reportDate, dayfirst=True)


Number appealCodes: 21, number reports: 21


C:\Users\lhasbini\AppData\Local\Temp\ipykernel_25116\71458790.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  reports_to_label_all[col_to_list] = reports_to_label_all[col_to_list].map(lambda x: listify_strings(x))


In [12]:
for i in range(len(reports_to_label_all)):
    nathaz_text = reports_to_label_all["nathaz_text"].iloc[i]
    # print(nathaz_text)
    print(f"Report {reports_to_label_all['appealCode'].iloc[i]}: {len(nathaz_text)} sentences")

Report MDR55001: 52 sentences
Report MDRAF014: 28 sentences
Report MDRCM036: 19 sentences
Report MDRDZ007: 53 sentences
Report MDRDZ008: 19 sentences
Report MDREC019: 47 sentences
Report MDRHT018: 74 sentences
Report MDRID013: 89 sentences
Report MDRIR009: 25 sentences
Report MDRJO003: 38 sentences
Report MDRKZ010: 56 sentences
Report MDRLB013: 30 sentences
Report MDRMN017: 52 sentences
Report MDRPG008: 33 sentences
Report MDRPH036: 53 sentences
Report MDRPK018: 48 sentences
Report MDRRS015: 25 sentences
Report MDRTJ035: 21 sentences
Report MDRTP004: 22 sentences
Report MDRUY004: 64 sentences
Report MDRVU012: 43 sentences


In [13]:
labelled_impact_reports_dict = {} # dict to store labelled reports
#empty dict structure to store results
# labelled_impact_reports_dict["appealCode"]=[
#     {"reportDate": None,
#      "impactSubtype" : None,
#      "impactValue" : None, 
#      "impactUnit" : None, 
#      "impactValuePrecision" : None, 
#      "impactValueMin" : None, 
#      "impactValueMax" : None, 
#      "annotation" : None,
#      "country" : None,
#      "location" : None,
#      "startYear" : None,
#      "startMonth" : None,
#      "startDay" : None,
#      "endYear" : None,
#      "endMonth" : None,
#      "endDay" : None,
#      "hazards" : None,
#     },
# ]

## Labelling 

In [14]:
i=0
print(f"{reports_to_label_all.iloc[i].appealCode}: {reports_to_label_all.iloc[i].reportDate}")
reports_to_label_all.iloc[i].nathaz_text

MDR55001: 2017-03-31


['International Appeal Operations update Pacific: Tropical Cyclone Pam International appeal n° MDR55001 GLIDE n° TC-2015-000020-VUT Operations update n° 7 (24-months consolidated report) Timeframe covered by this update: Issued on 31 March 2016 13 March 2015 to 28 February 2017 Operation start date: 13 March 2015 Operation end date: 30 September 2017 (30 months) Appeal budget (revised): CHF 8,024,577 Current expenditure: CHF 7,535,851 Appeal coverage: 100% donor response list Funding gap: CHF 0 Disaster Relief Emergency Fund (DREF) loan: CHF 575,000 Number of people affected: 247,408 Estimated number of people being assisted: 43,880 Operation Summary March-April 2015: Tropical Cyclone Pam developed in the East of Solomon Islands on 6 March and gained strength as it moved south towards Vanuatu.',
 'Heavy rain, strong winds and sea swell, combined with king tides, caused widespread coastal flooding and wind damage.',
 'Pam hit Vanuatu as a Category 5 cyclone on 13 March.',
 'An Informati

In [ ]:
labelled_impact_reports_dict["MDRBD022"]=[
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Affected People",
     "impactValue" : 2176519, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['DREF operation n MDRBD022 Glide n FL-2019-000079-BGD Date of issue 18 July 2019 Expected timeframe 4 months Expected end date 18 November 2019 Category allocated to the of the disaster or crisis Orange DREF allocated CHF 452,439 Total number of people affected 2,176,519 Number of people to be assisted 50,000 Host National Society presence (n of volunteers, staff, branches) Bangladesh Red Crescent Society (BDRCS) over 575 Red Crescent volunteers and 100 staff mobilized.'],
     "country" : ["Bangladesh"],
     "location" : None,
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 18,
     "endYear" : 2019,
     "endMonth" : 11,
     "endDay" : 18,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Affected People",
     "impactValue" : 1000000, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['While the monsoon season normally brings annual floods to the country and wider region, this year, widespread flooding in upstream countries, Nepal and India, where millions of people have been severely impacted, have meant that the scale of the flooding this year has been significantly exacerbated.'],
     "country" : ["Nepal", "India"],
     "location" : None,
     "startYear" : 2019,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 2100000, 
     "impactValueMax" : None, 
     "annotation" : ['According to National disaster response coordination centre (NDRCC) report dated 16 July more than 2.1 million people have been affected in 21 districts, around 100,000 houses destroyed and about 14,733 hectares of crop land damaged.', 
                     'Six days of heavy rain and onrush of upstream waters have flooded low-lying areas of Kurigram, Gaibandha, Lalmonirhat, Chattogram, Bandarban, Rangamati, Sylhet, Sunamganj, Bogura, Nilphamari, Khagrachari, Coxs Bazar, Feni, Netrokona, Sirajganj, Jamalpur, Tangail, Moulavibazar, Habiganj, Sherpur, and Brahmanbaria districts.'],
     "country" : ["Bangladesh"],
     "location" : ["Kurigram district", "Gaibandha district", "Lalmonirhat district", "Chattogram district", "Bandarban district", "Rangamati district", "Sylhet district", "Sunamganj district", "Bogura district", "Nilphamari district", "Khagrachari district", "Coxs Bazar district", "Feni district", "Netrokona district", "Sirajganj district", "Jamalpur district", "Tangail district", "Moulavibazar district", "Habiganj district", "Sherpur district", "Brahmanbaria district"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : 10000, 
     "impactUnit" : "houses destroyed", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['According to National disaster response coordination centre (NDRCC) report dated 16 July more than 2.1 million people have been affected in 21 districts, around 100,000 houses destroyed and about 14,733 hectares of crop land damaged.', 
                     'Six days of heavy rain and onrush of upstream waters have flooded low-lying areas of Kurigram, Gaibandha, Lalmonirhat, Chattogram, Bandarban, Rangamati, Sylhet, Sunamganj, Bogura, Nilphamari, Khagrachari, Coxs Bazar, Feni, Netrokona, Sirajganj, Jamalpur, Tangail, Moulavibazar, Habiganj, Sherpur, and Brahmanbaria districts.'],
     "country" : ["Bangladesh"],
     "location" : ["Kurigram district", "Gaibandha district", "Lalmonirhat district", "Chattogram district", "Bandarban district", "Rangamati district", "Sylhet district", "Sunamganj district", "Bogura district", "Nilphamari district", "Khagrachari district", "Coxs Bazar district", "Feni district", "Netrokona district", "Sirajganj district", "Jamalpur district", "Tangail district", "Moulavibazar district", "Habiganj district", "Sherpur district", "Brahmanbaria district"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Crop Production and Forestry",
     "impactValue" : 14733, 
     "impactUnit" : "hectares of crops", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['According to National disaster response coordination centre (NDRCC) report dated 16 July more than 2.1 million people have been affected in 21 districts, around 100,000 houses destroyed and about 14,733 hectares of crop land damaged.', 
                     'Six days of heavy rain and onrush of upstream waters have flooded low-lying areas of Kurigram, Gaibandha, Lalmonirhat, Chattogram, Bandarban, Rangamati, Sylhet, Sunamganj, Bogura, Nilphamari, Khagrachari, Coxs Bazar, Feni, Netrokona, Sirajganj, Jamalpur, Tangail, Moulavibazar, Habiganj, Sherpur, and Brahmanbaria districts.'],
     "country" : ["Bangladesh"],
     "location" : ["Kurigram district", "Gaibandha district", "Lalmonirhat district", "Chattogram district", "Bandarban district", "Rangamati district", "Sylhet district", "Sunamganj district", "Bogura district", "Nilphamari district", "Khagrachari district", "Coxs Bazar district", "Feni district", "Netrokona district", "Sirajganj district", "Jamalpur district", "Tangail district", "Moulavibazar district", "Habiganj district", "Sherpur district", "Brahmanbaria district"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Road Infrastructure",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['It is also reported that embankments have been damaged and inundated in new areas.', 
                     'Six days of heavy rain and onrush of upstream waters have flooded low-lying areas of Kurigram, Gaibandha, Lalmonirhat, Chattogram, Bandarban, Rangamati, Sylhet, Sunamganj, Bogura, Nilphamari, Khagrachari, Coxs Bazar, Feni, Netrokona, Sirajganj, Jamalpur, Tangail, Moulavibazar, Habiganj, Sherpur, and Brahmanbaria districts.'],
     "country" : ["Bangladesh"],
     "location" : ["Kurigram district", "Gaibandha district", "Lalmonirhat district", "Chattogram district", "Bandarban district", "Rangamati district", "Sylhet district", "Sunamganj district", "Bogura district", "Nilphamari district", "Khagrachari district", "Coxs Bazar district", "Feni district", "Netrokona district", "Sirajganj district", "Jamalpur district", "Tangail district", "Moulavibazar district", "Habiganj district", "Sherpur district", "Brahmanbaria district"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Affected People",
     "impactValue" : 2176519, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['of affected population 2,176,519 No.', 
                     'Six days of heavy rain and onrush of upstream waters have flooded low-lying areas of Kurigram, Gaibandha, Lalmonirhat, Chattogram, Bandarban, Rangamati, Sylhet, Sunamganj, Bogura, Nilphamari, Khagrachari, Coxs Bazar, Feni, Netrokona, Sirajganj, Jamalpur, Tangail, Moulavibazar, Habiganj, Sherpur, and Brahmanbaria districts.'],
     "country" : ["Bangladesh"],
     "location" : ["Kurigram district", "Gaibandha district", "Lalmonirhat district", "Chattogram district", "Bandarban district", "Rangamati district", "Sylhet district", "Sunamganj district", "Bogura district", "Nilphamari district", "Khagrachari district", "Coxs Bazar district", "Feni district", "Netrokona district", "Sirajganj district", "Jamalpur district", "Tangail district", "Moulavibazar district", "Habiganj district", "Sherpur district", "Brahmanbaria district"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : 3988, 
     "impactUnit" : "houses damaged", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['of fully damaged house 3,988 No.', 
                     'Six days of heavy rain and onrush of upstream waters have flooded low-lying areas of Kurigram, Gaibandha, Lalmonirhat, Chattogram, Bandarban, Rangamati, Sylhet, Sunamganj, Bogura, Nilphamari, Khagrachari, Coxs Bazar, Feni, Netrokona, Sirajganj, Jamalpur, Tangail, Moulavibazar, Habiganj, Sherpur, and Brahmanbaria districts.'],
     "country" : ["Bangladesh"],
     "location" : ["Kurigram district", "Gaibandha district", "Lalmonirhat district", "Chattogram district", "Bandarban district", "Rangamati district", "Sylhet district", "Sunamganj district", "Bogura district", "Nilphamari district", "Khagrachari district", "Coxs Bazar district", "Feni district", "Netrokona district", "Sirajganj district", "Jamalpur district", "Tangail district", "Moulavibazar district", "Habiganj district", "Sherpur district", "Brahmanbaria district"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : 98571, 
     "impactUnit" : "houses partially damaged", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['of partially damaged house 98,571 No.', 
                     'Six days of heavy rain and onrush of upstream waters have flooded low-lying areas of Kurigram, Gaibandha, Lalmonirhat, Chattogram, Bandarban, Rangamati, Sylhet, Sunamganj, Bogura, Nilphamari, Khagrachari, Coxs Bazar, Feni, Netrokona, Sirajganj, Jamalpur, Tangail, Moulavibazar, Habiganj, Sherpur, and Brahmanbaria districts.'],
     "country" : ["Bangladesh"],
     "location" : ["Kurigram district", "Gaibandha district", "Lalmonirhat district", "Chattogram district", "Bandarban district", "Rangamati district", "Sylhet district", "Sunamganj district", "Bogura district", "Nilphamari district", "Khagrachari district", "Coxs Bazar district", "Feni district", "Netrokona district", "Sirajganj district", "Jamalpur district", "Tangail district", "Moulavibazar district", "Habiganj district", "Sherpur district", "Brahmanbaria district"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Displaced People",
     "impactValue" : 31818, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['of people who have moved to safe shelter 31,818 Amount of crop land damaged (Hectare) 14,733 No.', 
                     'Six days of heavy rain and onrush of upstream waters have flooded low-lying areas of Kurigram, Gaibandha, Lalmonirhat, Chattogram, Bandarban, Rangamati, Sylhet, Sunamganj, Bogura, Nilphamari, Khagrachari, Coxs Bazar, Feni, Netrokona, Sirajganj, Jamalpur, Tangail, Moulavibazar, Habiganj, Sherpur, and Brahmanbaria districts.'],
     "country" : ["Bangladesh"],
     "location" : ["Kurigram district", "Gaibandha district", "Lalmonirhat district", "Chattogram district", "Bandarban district", "Rangamati district", "Sylhet district", "Sunamganj district", "Bogura district", "Nilphamari district", "Khagrachari district", "Coxs Bazar district", "Feni district", "Netrokona district", "Sirajganj district", "Jamalpur district", "Tangail district", "Moulavibazar district", "Habiganj district", "Sherpur district", "Brahmanbaria district"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Crop Production and Forestry",
     "impactValue" : 14733, 
     "impactUnit" : "hectares of crops", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['of people who have moved to safe shelter 31,818 Amount of crop land damaged (Hectare) 14,733 No.', 
                     'Six days of heavy rain and onrush of upstream waters have flooded low-lying areas of Kurigram, Gaibandha, Lalmonirhat, Chattogram, Bandarban, Rangamati, Sylhet, Sunamganj, Bogura, Nilphamari, Khagrachari, Coxs Bazar, Feni, Netrokona, Sirajganj, Jamalpur, Tangail, Moulavibazar, Habiganj, Sherpur, and Brahmanbaria districts.'],
     "country" : ["Bangladesh"],
     "location" : ["Kurigram district", "Gaibandha district", "Lalmonirhat district", "Chattogram district", "Bandarban district", "Rangamati district", "Sylhet district", "Sunamganj district", "Bogura district", "Nilphamari district", "Khagrachari district", "Coxs Bazar district", "Feni district", "Netrokona district", "Sirajganj district", "Jamalpur district", "Tangail district", "Moulavibazar district", "Habiganj district", "Sherpur district", "Brahmanbaria district"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Road Infrastructure",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['of union affected 776 Emergency Plan of Action (EPoA) Bangladesh Floods P a g e 2 People watch as water from the swollen Teesta river gush into their neighbourhood in Gaddimari area of Lalmonirhats Hatibandha after a part of the protection embankment collapsed on 13 July afternoon.'],
     "country" : ["Bangladesh"],
     "location" : ["Gaddimari area",  "Lalmonirhats Hatibandha"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 13,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }
]

In [ ]:
df_impact_list = []
for k,v in labelled_impact_reports_dict.items():
    df_impact = pd.DataFrame(v)
    df_impact['appealCode'] = k
    df_impact_list.append(df_impact)
df_impact_all = pd.concat(df_impact_list)
df_impact_all.reset_index(inplace=True, drop=True)

#Save impact csv 
fn = "labelled_reports_round2.csv" ##### CHANGE THE NAME OF THE REPORT TO SAVE
df_impact_all.to_csv(DATA_LABELLED+fn, index=False)